In [1]:
import os
import numpy as np
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

In [2]:
# === SETUP & DATA LOADING ===
SCRIPT_PATH = os.getcwd()
DATA_DIR = os.path.abspath(os.path.join(SCRIPT_PATH, '..', '..', 'data', 'preprocessed', 'time_window_data'))

print("Loading data...")
train_data = np.load(os.path.join(DATA_DIR, 'train.npz'))
val_data = np.load(os.path.join(DATA_DIR, 'val.npz'))
test_data = np.load(os.path.join(DATA_DIR, 'test.npz'))

# Concatenate all sets to ensure full evaluation coverage
X_full = np.concatenate([train_data['X'], val_data['X'], test_data['X']], axis=0)
y_full = np.concatenate([train_data['y'], val_data['y'], test_data['y']], axis=0)

# Load Label Encoder to get the number of classes
le = joblib.load(os.path.join(DATA_DIR, 'label_encoder.pkl'))
NUM_CLASSES = len(le.classes_)

Loading data...


In [3]:
# === HYPERPARAMETERS & CUSTOM LAYERS ===
INPUT_SHAPE = (X_full.shape[1], X_full.shape[2]) # (timesteps, features)
EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 0.001

class MCDropout(Dropout):
    """
    Custom Dropout layer that is ALWAYS active, even during inference.
    Required for Monte Carlo Uncertainty Estimation in future steps.
    """
    def call(self, inputs):
        return super().call(inputs, training=True)


In [4]:
#=== NESTED CROSS-VALIDATION (WITH EARLY STOPPING) ===
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
outer_acc, outer_prec, outer_rec, outer_f1 = [], [], [], []

print("\nStarting rigorous 5-Fold Cross-Validation for LSTM...")
print(f"Total samples: {len(X_full)}, Classes: {NUM_CLASSES}")
print("Note: Training 5 deep neural networks may take some time.\n")

fold_idx = 1
for train_idx, test_idx in outer_cv.split(X_full, y_full):
    print(f"--- Processing Fold {fold_idx}/5 ---")

    # Outer split: 80% for training pipeline, 20% for hidden testing
    X_outer_train, X_outer_test = X_full[train_idx], X_full[test_idx]
    y_outer_train, y_outer_test = y_full[train_idx], y_full[test_idx]

    # Inner split: Extract 20% of the training fold for Early Stopping validation
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    train_sub_idx, val_sub_idx = next(inner_cv.split(X_outer_train, y_outer_train))

    X_inner_train, X_inner_val = X_outer_train[train_sub_idx], X_outer_train[val_sub_idx]
    y_inner_train, y_inner_val = y_outer_train[train_sub_idx], y_outer_train[val_sub_idx]

    # Apply One-Hot Encoding strictly inside the fold
    y_inner_train_cat = to_categorical(y_inner_train, num_classes=NUM_CLASSES)
    y_inner_val_cat = to_categorical(y_inner_val, num_classes=NUM_CLASSES)

    # Build a fresh model architecture for each fold to prevent weight leakage
    model = Sequential([
        Input(shape=INPUT_SHAPE),
        LSTM(units=128, return_sequences=False),
        MCDropout(0.2),
        Dense(64, activation='relu'),
        MCDropout(0.2),
        Dense(NUM_CLASSES, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Early Stopping to prevent overfitting based on the inner validation set
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    # Train the model (verbose=0 to keep the console clean)
    model.fit(
        X_inner_train, y_inner_train_cat,
        validation_data=(X_inner_val, y_inner_val_cat),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stopping],
        verbose=0
    )

    # Evaluate the model on the completely unseen outer test set
    y_pred_probs = model.predict(X_outer_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Store metrics for the current fold
    outer_acc.append(accuracy_score(y_outer_test, y_pred))
    outer_prec.append(precision_score(y_outer_test, y_pred, average='macro'))
    outer_rec.append(recall_score(y_outer_test, y_pred, average='macro'))
    outer_f1.append(f1_score(y_outer_test, y_pred, average='macro'))

    fold_idx += 1


Starting rigorous 5-Fold Cross-Validation for LSTM...
Total samples: 147016, Classes: 4
Note: Training 5 deep neural networks may take some time.

--- Processing Fold 1/5 ---
--- Processing Fold 2/5 ---
--- Processing Fold 3/5 ---
--- Processing Fold 4/5 ---
--- Processing Fold 5/5 ---


In [5]:
# === 4. FINAL RESULTS OVERVIEW ===
print("\n=== FINAL RESULTS OF STRATIFIED CV: LSTM ===")
print(f"Accuracy:  {(np.mean(outer_acc) * 100):.1f}% ± {(np.std(outer_acc) * 100):.1f}%")
print(f"Precision: {(np.mean(outer_prec) * 100):.1f}% ± {(np.std(outer_prec) * 100):.1f}%")
print(f"Recall:    {(np.mean(outer_rec) * 100):.1f}% ± {(np.std(outer_rec) * 100):.1f}%")
print(f"F1-Score:  {(np.mean(outer_f1) * 100):.1f}% ± {(np.std(outer_f1) * 100):.1f}%")


=== FINAL RESULTS OF STRATIFIED CV: LSTM ===
Accuracy:  95.5% ± 1.7%
Precision: 95.5% ± 1.7%
Recall:    95.5% ± 1.7%
F1-Score:  95.5% ± 1.7%
